# Cooling Optimizer Demo

This notebook demonstrates the **CoolingOptimizerPlugin** from the EWIS toolkit.
The plugin evaluates ambient temperature, humidity, and Power Usage Effectiveness
(PUE) to produce a *cooling score* between 0 and 1, along with a recommended
cooling posture (`normal`, `elevated_cooling`, or `aggressive_cooling`).

We will:
1. Load a sample datacenter payload and run the plugin once.
2. Sweep across a range of temperature and humidity combinations to see how the
   score and posture change.
3. Isolate the effect of PUE on the recommendation.

Everything runs fully offline — no network calls required.

## Setup

We import standard-library utilities (`json`, `pathlib`), **pandas** for tabular
analysis, and the EWIS core classes needed to drive the plugin.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from ewis.core.context import PluginContext
from ewis.core.plugin_manager import PluginResult
from ewis.plugins.builtin.cooling_optimizer import CoolingOptimizerPlugin

## Load Sample Payload

The repository ships a ready-made JSON payload that mirrors a realistic
datacenter telemetry snapshot.  We load it from `../examples/data/sample_payload.json`.

In [ ]:
payload_path = Path("../examples/data/sample_payload.json")
payload = json.loads(payload_path.read_text())

print(json.dumps(payload, indent=2))

## Run the CoolingOptimizerPlugin

The plugin follows the standard EWIS lifecycle:

1. **Instantiate** — create the plugin with a logical name.
2. **Initialize** — pass a `PluginContext` so the plugin can read configuration.
3. **Execute** — feed the telemetry payload and receive a `PluginResult`.

The result includes `cooling_score` (0–1) and `recommended_posture`.

In [ ]:
ctx = PluginContext(config={})

plugin = CoolingOptimizerPlugin(name="cooling_optimizer")
plugin.initialize(ctx)

result: PluginResult = plugin.execute(payload, ctx)

print(f"ok            : {result.ok}")
print(f"cooling_score : {result.data['cooling_score']:.4f}")
print(f"posture       : {result.data['recommended_posture']}")
print(f"pue (metadata): {result.metadata['pue']}")

## Scenario Sweep: Temperature & Humidity

How does the cooling recommendation change as environmental conditions vary?
We sweep `ambient_temp_c` from 10 °C to 40 °C and `humidity_pct` from 20 % to
80 %, keeping PUE fixed at 1.24 (the value from our sample payload).  Each
combination is run through the plugin independently.

In [ ]:
temperatures = [10, 15, 20, 25, 30, 35, 40]
humidities = [20, 40, 60, 80]
fixed_pue = 1.24

rows = []
for temp in temperatures:
    for hum in humidities:
        scenario_payload = {
            "ambient_temp_c": temp,
            "humidity_pct": hum,
            "pue": fixed_pue,
        }
        res = plugin.execute(scenario_payload, ctx)
        rows.append({
            "ambient_temp_c": temp,
            "humidity_pct": hum,
            "pue": fixed_pue,
            "cooling_score": res.data["cooling_score"],
            "recommended_posture": res.data["recommended_posture"],
        })

df_sweep = pd.DataFrame(rows)
df_sweep

## Interpreting the Cooling Score

The plugin maps the continuous score to three discrete postures:

| Score range | Posture              | Meaning |
|-------------|----------------------|---------|
| ≤ 0.45      | `normal`             | Standard cooling is sufficient. |
| 0.45 – 0.75 | `elevated_cooling`   | Increase fan speed / lower set-points. |
| > 0.75      | `aggressive_cooling` | Maximum cooling — consider load shedding. |

Below we group the sweep results by posture and display a pivot table of scores
across temperature and humidity.

In [ ]:
posture_counts = df_sweep.groupby("recommended_posture").size().reset_index(name="count")
print("Posture distribution across scenarios:")
print(posture_counts.to_string(index=False))
print()

pivot = df_sweep.pivot_table(
    index="ambient_temp_c",
    columns="humidity_pct",
    values="cooling_score",
)
print("Cooling score by temperature (rows) and humidity (columns):")
pivot

## Effect of PUE on Recommendations

PUE (Power Usage Effectiveness) reflects the ratio of total facility energy to
IT equipment energy.  A PUE of 1.0 is ideal; higher values indicate overhead
from cooling, lighting, and other infrastructure.

The plugin adds a PUE-based penalty once PUE exceeds 1.25, pushing the cooling
score upward.  Here we fix temperature at 30 °C and humidity at 50 %, then vary
PUE from 1.1 to 1.8.

In [ ]:
pue_values = [round(1.1 + 0.1 * i, 1) for i in range(8)]  # 1.1 to 1.8

pue_rows = []
for pue in pue_values:
    scenario_payload = {
        "ambient_temp_c": 30,
        "humidity_pct": 50,
        "pue": pue,
    }
    res = plugin.execute(scenario_payload, ctx)
    pue_rows.append({
        "pue": pue,
        "cooling_score": res.data["cooling_score"],
        "recommended_posture": res.data["recommended_posture"],
    })

df_pue = pd.DataFrame(pue_rows)
df_pue

## Key Takeaways

* **Temperature is the dominant driver** — it contributes 70 % of the base score.
  Above ~28 °C most scenarios already land in `elevated_cooling` or higher.
* **Humidity adds a secondary push** — at 30 % of the base weight, high humidity
  alone rarely triggers aggressive cooling, but it can tip a borderline scenario.
* **PUE acts as a penalty offset** — only values above 1.25 increase the score,
  and the effect caps at +0.25.  Facilities with PUE ≤ 1.25 see no additional
  penalty from this factor.
* **Use the sweep table as a lookup** — operations teams can reference the pivot
  table to anticipate which weather forecasts will trigger posture changes and
  pre-position cooling capacity accordingly.
* **Extend the analysis** — try adding more PUE values, finer temperature steps,
  or integrating real weather-API data to build a live cooling dashboard.